# Fundamental Analysis — SEC 10-K/10-Q Report Generator

Given a list of stock tickers, this notebook fetches the latest SEC filings (10-K, 10-Q, 20-F, 6-K),
runs LLM-powered analysis on each filing section, and saves the results to an Excel file with
sentiment (positive / neutral / negative) per ticker.

In [ ]:
import sys
import pandas as pd
from collections import defaultdict
from tqdm import tqdm
from datetime import datetime

sys.path.append("../")

import importlib
import wallstreet_quant.edgar_pipeline
import wallstreet_quant.edgar_extractor

importlib.reload(wallstreet_quant.edgar_pipeline)
importlib.reload(wallstreet_quant.edgar_extractor)

from wallstreet_quant.edgar_pipeline import SecAnalysis
from wallstreet_quant.edgar_extractor import fetch_10K_and_10Q_filings

## Configuration

Edit the list below with the tickers you want to analyse.

In [ ]:
# ---------- USER CONFIG ----------
tickers = ["AAPL", "MSFT", "NVDA", "GOOGL", "AMZN"]  # <-- edit this list

model = "gpt-5.2-pro"          # LLM model for section analysis
latest_n = 2                    # number of filings to fetch per ticker (2 = current + previous for comparison)
output_filename = f"fundamental_analysis_{datetime.now().strftime('%m-%d-%Y')}.xlsx"
# ---------------------------------

print(f"Tickers: {tickers}")
print(f"Output:  {output_filename}")

## Fetch SEC Filings

Downloads the latest filings (10-K, 10-Q, 20-F, 6-K) for each ticker from SEC EDGAR.

In [ ]:
filings = defaultdict(list)

for ticker in tqdm(tickers, desc="Fetching filings"):
    try:
        filings[ticker] = fetch_10K_and_10Q_filings(ticker, latest_n=latest_n)
        print(f"  {ticker}: {len(filings[ticker])} filing(s) found")
    except Exception as e:
        print(f"  {ticker}: ERROR - {e}")

print(f"\nTotal: {sum(1 for v in filings.values() if v)} tickers with filings")

## Run LLM Analysis

For each ticker the pipeline analyses:
- **Risk Factors** (Item 1A) — added/removed risks vs previous filing
- **MD&A** (Item 7) — performance drivers, forward-looking statements
- **Legal Proceedings** (Item 3) — material litigation
- **Controls & Procedures** (Item 9A) — material weaknesses
- **Business Overview** (Item 1) — segments and changes
- **Tone Shift** — hedging/language changes between filings
- **Strategy & Guidance** — segment performance, priorities, guidance revisions
- **Human Capital & ESG** — headcount, retention, labor, ESG
- **Earnings Call** — latest call summary via web search

All results are consolidated into a final report with a **positive / neutral / negative** recommendation.

In [ ]:
sec_ai = SecAnalysis()
df = sec_ai(filings, model=model)
df

## Flatten Results for Excel

Expand nested dict columns into human-readable strings so the Excel file is easy to read.

In [ ]:
def flatten_risk_factors(d):
    if not isinstance(d, dict) or "error" in d:
        return d
    parts = []
    if d.get("changes_detected"):
        for r in d.get("added_risks", []):
            parts.append(f"[ADDED] {r['description']}")
        for r in d.get("removed_risks", []):
            parts.append(f"[REMOVED] {r['description']}")
    else:
        parts.append("No material changes detected")
    return "; ".join(parts) if parts else "N/A"


def flatten_mda(d):
    if not isinstance(d, dict) or "error" in d:
        return d
    drivers = ", ".join(d.get("performance_drivers", []))
    outlook = ", ".join(d.get("forward_looking_statements", []))
    opinion = d.get("honest_opinion", "")
    return f"Drivers: {drivers} | Outlook: {outlook} | Opinion: {opinion}"


def flatten_legal(d):
    if not isinstance(d, dict) or "error" in d:
        return d
    if not d.get("material_legal_proceedings"):
        return "No material legal proceedings"
    parts = []
    for p in d.get("proceedings", []):
        parts.append(f"{p['case']}: {p['description']} (Impact: {p['potential_impact']})")
    return "; ".join(parts)


def flatten_controls(d):
    if not isinstance(d, dict) or "error" in d:
        return d
    if not d.get("material_weaknesses_disclosed"):
        return "No material weaknesses"
    parts = [f"{w['area']}: {w['description']}" for w in d.get("weaknesses", [])]
    return "; ".join(parts)


def flatten_business(d):
    if not isinstance(d, dict) or "error" in d:
        return d
    segs = ", ".join(f"{s['name']} ({s['description']})" for s in d.get("segments", []))
    changes = ", ".join(d.get("segment_changes", []))
    return f"Segments: {segs} | Changes: {changes}"


def flatten_tone_shift(d):
    if not isinstance(d, dict) or "error" in d:
        return d
    if not d.get("language_shift_detected"):
        return "No significant tone shift"
    parts = [f"'{e['previous']}' -> '{e['current']}'" for e in d.get("examples", [])]
    return "; ".join(parts)


def flatten_strategy(d):
    if not isinstance(d, dict) or "error" in d:
        return d
    segs = "; ".join(
        f"{s['name']}: {s['performance']}" + (f" ({s['strategic_changes']})" if s.get('strategic_changes') else "")
        for s in d.get("segments", [])
    )
    priorities = ", ".join(d.get("corporate_priorities", []))
    guidance = "; ".join(
        f"{g['previous_guidance']} -> {g['revised_guidance']}" for g in d.get("guidance_revisions", [])
    )
    return f"Segments: {segs} | Priorities: {priorities} | Guidance: {guidance}"


def flatten_human_cap(d):
    if not isinstance(d, dict) or "error" in d:
        return d
    if not d.get("human_capital_changes"):
        return "No material human capital changes"
    parts = [f"{det['category']}: {det['description']}" for det in d.get("details", [])]
    return "; ".join(parts)


def flatten_earnings(d):
    if not isinstance(d, dict) or "error" in d:
        return d
    opinion = d.get("managements_opinion", "")
    risks = ", ".join(d.get("risks", []))
    growth = ", ".join(d.get("growth", []))
    buy = "Buy" if d.get("buy") else "Not Buy"
    return f"Opinion: {opinion} | Risks: {risks} | Growth: {growth} | Signal: {buy}"


# Build the flattened DataFrame
flat = pd.DataFrame()
flat["ticker"] = df["ticker"]
flat["recommendation"] = df["recommendation"]
flat["final_report"] = df["final_report"]
flat["risk_factors"] = df["risk_factors"].apply(flatten_risk_factors)
flat["md&a"] = df["md&a"].apply(flatten_mda)
flat["legal"] = df["legal"].apply(flatten_legal)
flat["controls"] = df["controls"].apply(flatten_controls)
flat["business"] = df["business"].apply(flatten_business)
flat["tone_shift"] = df["tone_shift"].apply(flatten_tone_shift)
flat["strategy_summary"] = df["strategy_summary"].apply(flatten_strategy)
flat["human_capital_esg"] = df["human_cap_esg"].apply(flatten_human_cap)
flat["earnings_call"] = df["earnings_call"].apply(flatten_earnings)

flat

## Save to Excel

In [ ]:
flat.to_excel(output_filename, index=False, engine="openpyxl")
print(f"Saved to {output_filename}")

## Summary

In [ ]:
if "recommendation" in flat.columns:
    print("Recommendation breakdown:")
    print(flat["recommendation"].value_counts().to_string())
    print()
    print("Positive:")
    pos = flat[flat["recommendation"] == "positive"]
    if len(pos):
        print("  " + ", ".join(pos["ticker"].tolist()))
    else:
        print("  None")
    print("Negative:")
    neg = flat[flat["recommendation"] == "negative"]
    if len(neg):
        print("  " + ", ".join(neg["ticker"].tolist()))
    else:
        print("  None")